# xBA Model

In this project, I will use statcast data from the pybaseball package with the goal of predicting the expected batting average (xBA) of a batted ball. I will use contact quality metrics like exit velocity, launch angle, and spray angle to train a classification model predicting the probability that a batted ball will be a hit. 

## Data Acquisition

To get the necessary pitch-level data, I will use the statcast function from the pybaseball package.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from pybaseball import statcast, playerid_reverse_lookup

START_DATE = '2023-03-30'
END_DATE = '2026-09-27'
TEST_START_DATE = '2026-03-25'
CACHE_DIR = Path('data')
CACHE_PATH = CACHE_DIR / f'statcast_{START_DATE}_{END_DATE}.parquet'

if CACHE_PATH.exists():
    raw = pd.read_parquet(CACHE_PATH)
    print(f'Loaded {len(raw):,} pitches from {CACHE_PATH}')
else:
    CACHE_DIR.mkdir(parents = True, exist_ok = True)
    raw = statcast(START_DATE, END_DATE)
    raw.to_parquet(CACHE_PATH, index = False)
    print(f'Downloaded and cached {len(raw):,} pitches to {CACHE_PATH}')

Loaded 3,018,395 pitches from data/statcast_2023-03-30_2026-09-27.parquet


In [2]:
bbe = {'field_out', 'single', 'double', 'home_run', 'force_out', 'grounded_into_double_play',
       'field_error', 'sac_fly', 'triple', 'fielders_choice', 'double_play', 'fielders_choice_out',
       'sac_fly_double_play', 'triple_play'}

contact = raw.loc[
    (raw.events.isin(bbe)) & (raw.game_type == 'R'),
    ['game_date', 'batter', 'events', 'launch_speed', 
     'launch_angle', 'stand', 'p_throws', 'pitch_type']]

contact['is_hit'] = contact["events"].isin(
    {"single", "double", "triple", "home_run"}
).astype(int)

contact.head(10)

,game_date,batter,events,launch_speed,launch_angle,stand,p_throws,pitch_type,is_hit
3,2026-09-27,811965,field_out,90.1,45,R,R,SL,0
4,2026-09-27,695720,field_out,68.5,46,R,R,SL,0
10,2026-09-27,664774,field_out,64.4,-53,L,L,ST,0
13,2026-09-27,665161,field_out,88.4,30,R,L,FF,0
19,2026-09-27,678489,field_out,79.6,82,R,L,FF,0
23,2026-09-27,643446,field_out,84.9,59,L,R,ST,0
24,2026-09-27,680869,field_out,93.3,43,R,R,SI,0
36,2026-09-27,671732,field_out,84.6,82,L,R,FF,0
39,2026-09-27,701358,force_out,72.9,-17,R,R,SL,0
52,2026-09-27,572233,single,62.4,-43,R,R,FF,1


In [3]:
model_data = contact.dropna(
    subset=["launch_speed", "launch_angle", "stand", "p_throws", "pitch_type", "is_hit"]
).copy()

model_data['launch_speed_x_launch_angle'] = model_data['launch_speed'] * model_data['launch_angle']

print(f"Contact events: {len(contact):,}")
print(f"Complete rows: {len(model_data):,}")
print(f"Observed batting average: {model_data['is_hit'].mean():.3f}")
print("Missing values by field:")
print(contact.isna().sum().sort_values(ascending=False))

Contact events: 495,813
Complete rows: 494,102
Observed batting average: 0.324
Missing values by field:
launch_speed    1659
launch_angle    1358
pitch_type       187
game_date          0
batter             0
events             0
stand              0
p_throws           0
is_hit             0
dtype: int64


In [4]:
model_data.game_date = pd.to_datetime(model_data.game_date)
test_start_date = pd.Timestamp(TEST_START_DATE)
train_data = model_data[model_data['game_date'] < test_start_date].copy()
test_data = model_data[model_data['game_date'] >= test_start_date].copy()

In [5]:
numeric_features = ['launch_speed', 'launch_angle', 'launch_speed_x_launch_angle']
categorical_features = ['stand', 'p_throws', 'pitch_type']
features = numeric_features + categorical_features

X_train = train_data[features]
X_test = test_data[features]
y_train = train_data['is_hit']
y_test = test_data['is_hit']

print(f"Training rows: {len(train_data):,}")
print(f"Test rows: {len(test_data):,}")
print(f"Training dates: {train_data['game_date'].min().date()} to {train_data['game_date'].max().date()}")
print(f"Test dates: {test_data['game_date'].min().date()} to {test_data['game_date'].max().date()}")

Training rows: 370,720
Test rows: 123,382
Training dates: 2023-03-30 to 2025-09-28
Test dates: 2026-03-25 to 2026-09-27


## Model Training

I now have a dataframe will batted balls from 2023-2026. With this dataframe, I will train a classification model to predict xBA. I will start with a dummy classifier as a baseline, and compare the performance of more complex models, including Logistic Regression, Spline Logistic Regression, Random Forest Classifier, and XGBoost Classifier. 

I will train my models on data from 2023-2025 and the 2026 data will be used for testing. To maximize the performance of my models, I will use GridSearchCV for hyperparameter optimization, validating on the 2025 data. To evaluate my models, I will use the log-loss score. 

In [6]:
#Baseline model
from sklearn.dummy import DummyClassifier

baseline = DummyClassifier(strategy = 'prior')
baseline.fit(X_train, y_train)
baseline_probs = baseline.predict_proba(X_test)[:, 1]

In [7]:
#Simple Logistic Regression model
#One-hot encoding for categorical features

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score, brier_score_loss, log_loss

preprocessor = ColumnTransformer(
    transformers = [
        ('numeric', StandardScaler(), numeric_features),
        ('categorical', OneHotEncoder(handle_unknown = 'ignore'), categorical_features)
    ]
)

logistic_model = Pipeline(
    steps = [
        ('preprocessor', preprocessor),
        ('model', LogisticRegression(random_state = 99))
    ]
)

logistic_model.fit(X_train, y_train)
logistic_model_probs = logistic_model.predict_proba(X_test)[:, 1]

metrics = pd.DataFrame({
    'log_loss': [
        log_loss(y_test, baseline_probs),
        log_loss(y_test, logistic_model_probs)
    ],

    'brier_score': [
        brier_score_loss(y_test, baseline_probs),
        brier_score_loss(y_test, logistic_model_probs)
    ],

    'accuracy_at_0.5': [
        accuracy_score(y_test, baseline_probs >= 0.5),
        accuracy_score(y_test, logistic_model_probs >= 0.5)
    ]
}, index = ['Baseline Classifier', 'Logistic Model'])

metrics

,log_loss,brier_score,accuracy_at_0.5
Baseline Classifier,0.628487,0.218372,0.677870
Logistic Model,0.594849,0.201166,0.703287


In [8]:
#Spline Logistic Model with Interaction Effects

from sklearn.preprocessing import SplineTransformer
from sklearn.model_selection import GridSearchCV

spline_preprocessor = ColumnTransformer(
    transformers = [
        ('splines', SplineTransformer(degree = 3, knots = 'quantile'), numeric_features),
        ('onehot', OneHotEncoder(handle_unknown = 'ignore'), categorical_features)
    ]
)

spline_model = Pipeline(
    steps = [
        ('preprocessor', spline_preprocessor),
        ('model', LogisticRegression(max_iter = 1000, random_state = 99))
    ]
)

spline_param_grid = {
    'preprocessor__splines__n_knots': [3, 5, 8, 12, 16],
    'model__C': [0.01, 0.1, 1, 10, 100]
}

seasons = train_data['game_date'].dt.year.to_numpy()
train_idx = np.where(seasons < 2025)[0]
val_idx = np.where(seasons == 2025)[0]

spline_grid = GridSearchCV(
    estimator = spline_model,
    param_grid = spline_param_grid,
    scoring = 'neg_log_loss',
    n_jobs = 1,
    cv = [(train_idx, val_idx)],
)

spline_grid.fit(X_train, y_train)
spline_grid.best_params_

{'model__C': 100, 'preprocessor__splines__n_knots': 16}

In [9]:
spline_model = spline_grid.best_estimator_
spline_model_probs = spline_model.predict_proba(X_test)[:, 1]

metrics.loc['Spline Logistic Model'] = [
    log_loss(y_test, spline_model_probs),
    brier_score_loss(y_test, spline_model_probs),
    accuracy_score(y_test, spline_model_probs >= 0.5)
]

metrics

,log_loss,brier_score,accuracy_at_0.5
Baseline Classifier,0.628487,0.218372,0.677870
Logistic Model,0.594849,0.201166,0.703287
Spline Logistic Model,0.450068,0.145604,0.796867


In [10]:
#Random Forest Classifier
from sklearn.ensemble import RandomForestClassifier

rf_numeric = ['launch_speed', 'launch_angle']
rf_features = rf_numeric + categorical_features

rf_preprocessor = ColumnTransformer(
    transformers = [
        ('numeric', 'passthrough', rf_numeric),
        ('onehot', OneHotEncoder(handle_unknown = 'ignore'), categorical_features)
    ]
)

rf_model = Pipeline(
    steps = [
        ('preprocessor', rf_preprocessor),
        ('model', RandomForestClassifier(
            n_estimators = 100, 
            max_features = 1.0,
            n_jobs = -1, 
            random_state = 99))
    ]
)

rf_param_grid = {
    'model__min_samples_leaf': [50, 100, 150],
    'model__max_samples': [0.5, 0.75, None]
}

subset = train_data.sample(frac = 0.5, random_state = 99)
X_sub = subset[rf_features]
y_sub = subset['is_hit']

seasons = subset['game_date'].dt.year.to_numpy()
train_idx = np.where(seasons < 2025)[0]
val_idx = np.where(seasons == 2025)[0]

rf_grid = GridSearchCV(
    estimator = rf_model,
    param_grid = rf_param_grid,
    scoring = 'neg_log_loss',
    n_jobs = 1,
    cv = [(train_idx, val_idx)],
    refit = False
)

rf_grid.fit(X_sub, y_sub)
rf_grid.best_params_

{'model__max_samples': 0.5, 'model__min_samples_leaf': 100}

In [11]:
X_train_rf = train_data[rf_features]
X_test_rf = test_data[rf_features]

rf_model.set_params(**rf_grid.best_params_)
rf_model.fit(X_train_rf, y_train)
rf_model_probs = rf_model.predict_proba(X_test_rf)[:, 1]

metrics.loc['Random Forest Classifier'] = [
    log_loss(y_test, rf_model_probs),
    brier_score_loss(y_test, rf_model_probs),
    accuracy_score(y_test, rf_model_probs >= 0.5)
]

metrics

,log_loss,brier_score,accuracy_at_0.5
Baseline Classifier,0.628487,0.218372,0.677870
Logistic Model,0.594849,0.201166,0.703287
Spline Logistic Model,0.450068,0.145604,0.796867
Random Forest Classifier,0.405898,0.131271,0.808530


In [12]:
#Gradient Boosting Model

from xgboost import XGBClassifier

xgb_model = Pipeline(
    steps = [
        ('preprocessor', rf_preprocessor),
        ('model', XGBClassifier(
            objective = 'binary:logistic',
            eval_metric = 'logloss',
            learning_rate = 0.05,
            n_estimators = 500,
            max_depth = 5,
            subsample = 0.8,
            colsample_bytree = 0.8,
            n_jobs = -1,
            random_state = 99
        ))
    ]
)

xgb_param_grid = {
    'model__max_depth': [3, 5, 7],
    'model__learning_rate': [0.05, 0.1, 0.15],
    'model__n_estimators': [200, 300, 400],
    'model__subsample': [0.6, 0.8, 1.0]
}

xgb_grid = GridSearchCV(
    estimator = xgb_model,
    param_grid = xgb_param_grid,
    scoring = 'neg_log_loss',
    n_jobs = 1,
    cv = [(train_idx, val_idx)],
    refit = False
)

xgb_grid.fit(X_sub, y_sub)
xgb_grid.best_params_

{'model__learning_rate': 0.1,
 'model__max_depth': 5,
 'model__n_estimators': 300,
 'model__subsample': 0.8}

In [13]:
xgb_model.set_params(**xgb_grid.best_params_)
xgb_model.fit(X_train_rf, y_train)
xgb_model_probs = xgb_model.predict_proba(X_test_rf)[:, 1]

metrics.loc['XGB Classifier'] = [
    log_loss(y_test, xgb_model_probs),
    brier_score_loss(y_test, xgb_model_probs),
    accuracy_score(y_test, xgb_model_probs >= 0.5)
]

metrics

,log_loss,brier_score,accuracy_at_0.5
Baseline Classifier,0.628487,0.218372,0.677870
Logistic Model,0.594849,0.201166,0.703287
Spline Logistic Model,0.450068,0.145604,0.796867
Random Forest Classifier,0.405898,0.131271,0.808530
XGB Classifier,0.405484,0.131056,0.809089


## Batter Summaries

I will use my XGBoost model to predict the xBA for each batter during the 2026 season. Strikeouts or at-bats not ending in a ball-in-play will get an xBA value of .000. 

In [14]:
hit = {'single', 'double', 'triple', 'home_run'}
atbat = {'single', 'double', 'triple', 'home_run', 'field_out', 'strikeout', 'force_out', 'grounded_into_double_play',
          'field_error', 'fielders_choice', 'double_play', 'fielders_choice_out', 'strikeout_double_play', 'triple_play'}

raw.game_date = pd.to_datetime(raw.game_date)
seasons = raw['game_date'].dt.year.to_numpy()
data2026 = raw.loc[(seasons == 2026) & (raw.game_type == 'R') & (raw.events.isin(atbat))][['game_date', 'batter', 'events'] + rf_features].reset_index(drop = True)

data2026.head(10)

,game_date,batter,events,launch_speed,launch_angle,stand,p_throws,pitch_type
0,2026-09-27,672016,strikeout,<NA>,<NA>,R,R,SL
1,2026-09-27,811965,field_out,90.1,45,R,R,SL
2,2026-09-27,695720,field_out,68.5,46,R,R,SL
3,2026-09-27,664774,field_out,64.4,-53,L,L,ST
4,2026-09-27,665161,field_out,88.4,30,R,L,FF
5,2026-09-27,678489,field_out,79.6,82,R,L,FF
6,2026-09-27,643446,field_out,84.9,59,L,R,ST
7,2026-09-27,680869,field_out,93.3,43,R,R,SI
8,2026-09-27,671732,field_out,84.6,82,L,R,FF
9,2026-09-27,701358,force_out,72.9,-17,R,R,SL


In [15]:
raw.events.value_counts()

events
field_out                    311037
strikeout                    174107
single                       110268
walk                          64279
double                        33144
home_run                      23862
force_out                     14627
grounded_into_double_play     13721
hit_by_pitch                   8670
sac_fly                        5326
field_error                    4566
triple                         2895
sac_bunt                       2214
intent_walk                    2160
fielders_choice                1648
double_play                    1575
truncated_pa                   1318
fielders_choice_out            1300
strikeout_double_play           458
catcher_interf                  392
sac_fly_double_play              69
triple_play                       7
Name: count, dtype: int64

In [16]:
data2026.isna().sum()

game_date           0
batter              0
events              0
launch_speed    41199
launch_angle    41130
stand               0
p_throws            0
pitch_type        112
dtype: int64

In [17]:
is_bbe = data2026['events'].isin(bbe)
has_missing = data2026.isna().any(axis = 1)
strikeout = data2026['events'] == 'strikeout'
data2026['is_hit'] = data2026['events'].isin(hit).astype('int')

data2026.loc[(~has_missing | strikeout), 'xba'] = 0.0

data2026.loc[(is_bbe & ~has_missing), 'xba'] = xgb_model.predict_proba(data2026.loc[(is_bbe & ~has_missing), rf_features])[:, 1]

player_ids = (
    playerid_reverse_lookup(data2026['batter'].unique())
    [['key_mlbam', 'name_last', 'name_first']]
    .rename(columns = {
        'key_mlbam': 'batter',
        'name_last': 'last_name',
        'name_first': 'first_name'
    })
)

data2026 = data2026.merge(player_ids, on = 'batter', how = 'left')

data2026.head(10)

Gathering player lookup table. This may take a moment.


,game_date,batter,events,launch_speed,launch_angle,stand,p_throws,pitch_type,is_hit,xba,last_name,first_name
0,2026-09-27,672016,strikeout,<NA>,<NA>,R,R,SL,0,0.000000,clarke,denzel
1,2026-09-27,811965,field_out,90.1,45,R,R,SL,0,0.004026,kuroda-grauer,joshua
2,2026-09-27,695720,field_out,68.5,46,R,R,SL,0,0.268466,white,tommy
3,2026-09-27,664774,field_out,64.4,-53,L,L,ST,0,0.188010,wade,lamonte
4,2026-09-27,665161,field_out,88.4,30,R,L,FF,0,0.048493,peña,jeremy
5,2026-09-27,678489,field_out,79.6,82,R,L,FF,0,0.001830,barrosa,jorge
6,2026-09-27,643446,field_out,84.9,59,L,R,ST,0,0.020402,mcneil,jeff
7,2026-09-27,680869,field_out,93.3,43,R,R,SI,0,0.010401,gelof,zack
8,2026-09-27,671732,field_out,84.6,82,L,R,FF,0,0.002178,butler,lawrence
9,2026-09-27,701358,force_out,72.9,-17,R,R,SL,0,0.057962,smith,cam


In [18]:
batter_summary = (
    data2026
    .groupby(['batter'], as_index = False)
    .agg(
        last_name = ('last_name', 'first'),
        first_name = ('first_name', 'first'),
        at_bats = ('is_hit', 'size'),
        hits = ('is_hit', 'sum'),
        avg = ('is_hit', 'mean'),
        xBA = ('xba', 'mean')
    )
    .sort_values(by = 'at_bats', ascending = False)
)

batter_summary[['avg', 'xBA']] = batter_summary[['avg', 'xBA']].round(3)
batter_summary['avg - xBA'] = batter_summary['avg'] - batter_summary['xBA']
print("2026 Batter Summary")
batter_summary.head(10)

2026 Batter Summary


,batter,last_name,first_name,at_bats,hits,avg,xBA,avg - xBA
60,607208,turner,trea,645,156,0.242,0.232,0.010
230,666182,bichette,bo,638,166,0.260,0.285,-0.025
164,663538,hoerner,nico,628,165,0.263,0.292,-0.029
326,672640,lopez,otto,627,191,0.305,0.274,0.031
516,691718,crow-armstrong,pete,626,175,0.280,0.258,0.022
204,665487,tatís,fernando,626,181,0.289,0.293,-0.004
75,621566,olson,matt,625,157,0.251,0.252,-0.001
594,701398,stewart,sal,623,164,0.263,0.259,0.004
442,683002,henderson,gunnar,623,140,0.225,0.239,-0.014
477,687263,neto,zach,619,148,0.239,0.215,0.024


In [19]:
print('"Unluckiest" Batters in 2026 by AVG - xBA (>300 ABs)')
batter_summary.loc[batter_summary['at_bats'] > 200].sort_values(by = 'avg - xBA').head(10)

"Unluckiest" Batters in 2026 by AVG - xBA (>300 ABs)


,batter,last_name,first_name,at_bats,hits,avg,xBA,avg - xBA
172,663647,hayes,ke'bryan,227,32,0.141,0.240,-0.099
180,663837,vierling,matt,266,55,0.207,0.249,-0.042
234,666397,julien,edouard,236,49,0.208,0.250,-0.042
36,592663,realmuto,j. t.,414,85,0.205,0.246,-0.041
317,672275,bailey,patrick,310,64,0.206,0.246,-0.040
220,665966,narváez,carlos,248,40,0.161,0.195,-0.034
179,663757,grisham,trent,418,88,0.211,0.243,-0.032
61,608070,ramírez,josé,473,111,0.235,0.265,-0.030
64,608369,seager,corey,384,87,0.227,0.257,-0.030
11,521692,pérez,salvador,580,133,0.229,0.258,-0.029


In [ ]:
print('"Luckiest" Batters in 2026 by AVG - xBA (>300 ABs)')
batter_summary.loc[batter_summary['at_bats'] > 200].sort_values(by = 'avg - xBA', ascending = False).head(10)

"Luckiest" Batters in 2026 by AVG - xBA (>300 ABs)


,batter,last_name,first_name,at_bats,hits,avg,xBA,avg - xBA
620,802415,simpson,chandler,571,175,0.306,0.247,0.059
341,675961,williams,alika,215,58,0.270,0.215,0.055
217,665923,ruiz,esteury,213,46,0.216,0.170,0.046
342,676356,deluca,jonny,350,93,0.266,0.222,0.044
613,703607,bolte,henry,415,115,0.277,0.237,0.040
383,678882,rafaela,ceddanne,541,154,0.285,0.245,0.040
184,663968,mangum,jake,435,120,0.276,0.238,0.038
641,807712,keaschall,luke,528,149,0.282,0.244,0.038
457,685133,meckler,wade,300,90,0.300,0.262,0.038
466,686668,doyle,brenton,210,49,0.233,0.196,0.037
